In [1]:
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import pysam
import cvxpy as cp

from utils import filename2ctype

%load_ext autoreload
%autoreload 2

## Data Gathering

In [2]:
PATH_TO_ROOT_FOLDER = Path(
    "/staging/leuven/stg_00118/methylDL/data/loyfer2023/hg38/data/GSE186458"
)

In [3]:
ctype_to_sample_file_names = defaultdict(list)
for f in PATH_TO_ROOT_FOLDER.glob("*.pat.gz"):
    ctype_name = filename2ctype(f.name)
    if ctype_name:
        ctype_to_sample_file_names[ctype_name].append(f.name)
ctype_to_sample_file_names = dict(ctype_to_sample_file_names)

### Blood file

In [4]:
blood_file_path = (
    PATH_TO_ROOT_FOLDER / ctype_to_sample_file_names["Blood-Mono+Macro"][0]
)
blood_file_path

PosixPath('/staging/leuven/stg_00118/methylDL/data/loyfer2023/hg38/data/GSE186458/GSM5652311_Lung-Interstitial-Macrophages-Z0000044D.hg38.pat.gz')

In [5]:
blood_df = pd.read_csv(
    blood_file_path,
    sep="\t",
    header=None,
    compression="infer",
    nrows=10000,
    skiprows=1_000_000,
)
blood_df.columns = ["chr", "index", "methyl", "n_reads"]
print(
    f"Start index range in blood sample: {blood_df['index'].min()} - {blood_df['index'].max()}"
)

Start index range in blood sample: 232173 - 234095


In [6]:
if blood_df.chr.nunique() > 1:
    raise ValueError(
        "Multiple chromosomes found in the blood sample. Please choose a different region for analysis."
    )
chromosome_under_study = blood_df.chr.unique()[0]
print(f"Chromosome under study: {chromosome_under_study}")

Chromosome under study: chr1


Interesting values:
- {n_rows= 10000, N_CPGS=8, skiprows=1_000_000}

In [7]:
# we are looking through the file and looking for a region containing N_CPGS
# with the maximum number of reads covering all the CpGs in the region
N_CPGS = 5
# a region is uniquely identified by a start position in CpG-order and the number of CpGs
unique_index = blood_df[
    "index"
].unique()  # no need to sort as the file is already sorted by index
index_to_pos = {idx: pos for pos, idx in enumerate(unique_index)}
n_unique_indices = len(unique_index)
n_regions = n_unique_indices - N_CPGS + 1
read_count_per_region = np.zeros(n_regions, dtype=np.int32)
for row in blood_df.itertuples():
    start_index = row.index
    n_reads = row.n_reads
    n_cpgs = len(row.methyl)
    if n_cpgs >= N_CPGS:
        end_index = min(start_index + (n_cpgs - N_CPGS + 1), unique_index[-1])
        while end_index not in index_to_pos:
            end_index -= 1
        end_pos = min(index_to_pos[end_index], len(read_count_per_region))
        read_count_per_region[index_to_pos[start_index] : end_pos] += n_reads

In [8]:
max_count_region_idx = np.argmax(read_count_per_region)
max_count_region_start_index = unique_index[max_count_region_idx]
max_count = read_count_per_region[max_count_region_idx]
max_count, max_count_region_start_index, min(unique_index), max(unique_index)

(np.int32(70), np.int64(232349), np.int64(232173), np.int64(234095))

In [9]:
# now we retrieve the counts of all the signatures in the region
blood_sig_counts = defaultdict(int)
region_start_pos = max_count_region_idx
region_end_pos_excl = region_start_pos + N_CPGS
for row in blood_df.itertuples():
    start_pos = index_to_pos[row.index]
    n_reads = row.n_reads
    methyl = row.methyl
    n_cpgs = len(methyl)
    read_end_pos_excl = start_pos + n_cpgs
    # keep only reads that fully cover this N_CPGS region in CpG-order space
    if start_pos <= region_start_pos and read_end_pos_excl >= region_end_pos_excl:
        start_overlap = region_start_pos - start_pos
        end_overlap = region_end_pos_excl - start_pos
        extracted_pattern = methyl[start_overlap:end_overlap]
        if (
            "." not in extracted_pattern
        ):  # we only consider patterns that have no missing values
            blood_sig_counts[extracted_pattern] += n_reads
blood_sig_counts = dict(blood_sig_counts)
blood_sig_counts

{'TTTTT': 63, 'TCTTC': 1, 'CTCTT': 1, 'CTTTT': 1}

### Neuron file

Now we retrieve the signature counts for the same region for another cell type (here: neuron)

In [10]:
neuron_file_path = PATH_TO_ROOT_FOLDER / ctype_to_sample_file_names["Neuron"][0]
neuron_file_path

PosixPath('/staging/leuven/stg_00118/methylDL/data/loyfer2023/hg38/data/GSE186458/GSM5652229_Cortex-Neuron-Z0000042P.hg38.pat.gz')

We need to load the area around the region selected for the blood file.

In [11]:
# Load the CSI index file to determine the correct offset
csi_file_path = Path("./data/GSM5652229_Cortex-Neuron-Z0000042P.hg38.pat.gz.csi")

# Extract byte offset from CSI index for the target region

# pysam can read the CSI index to get the file offset for a specific genomic region
# We'll use it to find the approximate starting position
tbx = pysam.TabixFile(str(neuron_file_path), index=str(csi_file_path))
# Get the byte offset for the chromosome and start position
# For now, we'll fetch a region around the target start index
neuron_df_list = []
for row in tbx.fetch(
    chromosome_under_study,
    max_count_region_start_index - 50,
    max_count_region_start_index + (N_CPGS + 50),
):
    parts = row.split("\t")
    neuron_df_list.append(
        {
            "chr": parts[0],
            "index": int(parts[1]),
            "methyl": parts[2],
            "n_reads": int(parts[3]),
        }
    )
neuron_df = pd.DataFrame(neuron_df_list)

print(f"Chromosomes: {neuron_df['chr'].unique()}")
print(f"Min index: {neuron_df['index'].min()}, max index: {neuron_df['index'].max()}")
print(f"Target region start index: {max_count_region_start_index}")

Chromosomes: ['chr1']
Min index: 232301, max index: 232404
Target region start index: 232349


In [12]:
second_neuron_df = pd.read_csv(
    neuron_file_path,
    sep="\t",
    header=None,
    compression="infer",
    nrows=10000,
    skiprows=1_000_000,
)
print(f"Chromosomes: {second_neuron_df[0].unique()}")
print(f"Min index: {second_neuron_df[1].min()}, max index: {second_neuron_df[1].max()}")
print(f"Target region start index: {max_count_region_start_index}")

Chromosomes: ['chr1']
Min index: 288051, max index: 290633
Target region start index: 232349


In [13]:
# now we retrieve the counts of all the signatures in the region
neuron_sig_counts = defaultdict(int)
neuron_unique_index = neuron_df[
    "index"
].unique()  # no need to sort as the file is already sorted by index
neuron_index_to_pos = {idx: pos for pos, idx in enumerate(neuron_unique_index)}
region_start_pos = neuron_index_to_pos[max_count_region_start_index]
region_end_pos_excl = region_start_pos + N_CPGS
for row in neuron_df.itertuples():
    start_pos = neuron_index_to_pos[row.index]
    n_reads = row.n_reads
    methyl = row.methyl
    n_cpgs = len(methyl)
    read_end_pos_excl = start_pos + n_cpgs
    # keep only reads that fully cover this N_CPGS region in CpG-order space
    if start_pos <= region_start_pos and read_end_pos_excl >= region_end_pos_excl:
        start_overlap = region_start_pos - start_pos
        end_overlap = region_end_pos_excl - start_pos
        extracted_pattern = methyl[start_overlap:end_overlap]
        # we only consider patterns that have no missing values
        if "." not in extracted_pattern:
            neuron_sig_counts[extracted_pattern] += n_reads
neuron_sig_counts = dict(neuron_sig_counts)
neuron_sig_counts

{'TTTTT': 32,
 'CCCCT': 1,
 'CTCTT': 1,
 'TTCCT': 1,
 'TTCTT': 2,
 'TTTCT': 1,
 'CTTTT': 1}

## Modelling

In [14]:
##
cell_type_sig_counts = {
    "Blood-Mono+Macro": blood_sig_counts,
    "Neuron": neuron_sig_counts,
}
all_signatures = set()
for sig_counts in cell_type_sig_counts.values():
    all_signatures.update(sig_counts.keys())
all_signatures = sorted(all_signatures)

sig_count_df = pd.DataFrame(
    {
        ctype: [sig_counts.get(sig, 0) for sig in all_signatures]
        for ctype, sig_counts in cell_type_sig_counts.items()
    },
    index=all_signatures,
)
sig_proportions_df = sig_count_df.div(sig_count_df.sum(axis=0), axis=1)
print("Signature proportions for each cell type:")
sig_proportions_df.round(2)

Signature proportions for each cell type:


,Blood-Mono+Macro,Neuron
CCCCT,0.00,0.03
CTCTT,0.02,0.03
CTTTT,0.02,0.03
TCTTC,0.02,0.00
TTCCT,0.00,0.03
TTCTT,0.00,0.05
TTTCT,0.00,0.03
TTTTT,0.95,0.82


In [15]:
print("Signature counts for each cell type:")
sig_count_df

Signature counts for each cell type:


,Blood-Mono+Macro,Neuron
CCCCT,0,1
CTCTT,1,1
CTTTT,1,1
TCTTC,1,0
TTCCT,0,1
TTCTT,0,2
TTTCT,0,1
TTTTT,63,32


In [16]:
# now simulate several mixted samples with different mixing proportions
pseudo_samples_mixing_proportions = [
    0.1,
    0.25,
    0.5,
    0.3,
]  # it indicates the proportion of "Blood" in the mixture
pseudo_samples_read_counts = [50, 50, 50, 50]  # total read count for each pseudo sample
pseudo_samples = [
    np.concatenate(
        [
            np.random.choice(
                sig_count_df.index,
                size=int(mix_prop * total_reads),
                p=sig_count_df["Blood-Mono+Macro"]
                / sig_count_df["Blood-Mono+Macro"].sum(),
            ),
            np.random.choice(
                sig_count_df.index,
                size=int((1 - mix_prop) * total_reads),
                p=sig_count_df["Neuron"] / sig_count_df["Neuron"].sum(),
            ),
        ]
    )
    for mix_prop, total_reads in zip(
        pseudo_samples_mixing_proportions, pseudo_samples_read_counts
    )
]
pseudo_sample_sig_counts = []
for sample in pseudo_samples:
    sig_counts = defaultdict(int)
    for sig in sample:
        sig_counts[sig] += 1
    pseudo_sample_sig_counts.append(dict(sig_counts))
pseudo_sample_sig_counts_df = (
    pd.DataFrame(pseudo_sample_sig_counts).fillna(0).astype(int).T.sort_index()
)
pseudo_sample_sig_proportions_df = (
    pseudo_sample_sig_counts_df / pseudo_sample_sig_counts_df.sum()
)

In [17]:
# for visualization purposes, we create a df with the proportions of each ctype in the sample
vis_df = pseudo_sample_sig_proportions_df.copy().T
vis_df["Blood proportion"] = pseudo_samples_mixing_proportions
vis_df["Neuron proportion"] = [1 - prop for prop in pseudo_samples_mixing_proportions]
print("Signature and ctype proportions for each pseudo sample:")
vis_df.T.round(2)

Signature and ctype proportions for each pseudo sample:


,0,1,2,3
CCCCT,0.00,0.04,0.00,0.02
CTCTT,0.06,0.00,0.02,0.02
CTTTT,0.04,0.02,0.00,0.04
TTCCT,0.02,0.02,0.04,0.00
TTCTT,0.00,0.08,0.06,0.00
TTTCT,0.04,0.02,0.00,0.04
TTTTT,0.84,0.82,0.88,0.88
Blood proportion,0.10,0.25,0.50,0.30
Neuron proportion,0.90,0.75,0.50,0.70


Now onto the least square deconvolution.

Let $M$ be the matrix of signature frequencies per sample (shape $N_S \times N_{sig}$),
$P$ be the matrix of cell type proportions per sample (shape $N_S \times C$),
and $F$ be the matrix of signature frequencies per cell type (shape $C \times N_{sig}$).
$F$ is the unknown we are solving for.

In [18]:
F = cp.Variable(
    (2, len(all_signatures)), nonneg=True
)  # shape: (number of cell types, number of signatures)
M = pseudo_sample_sig_counts_df.values.T.astype(
    float
)  # shape: (number of pseudo samples, number of signatures)
M /= M.sum(axis=1, keepdims=True)  # convert to frequencies
P = np.array(
    [
        pseudo_samples_mixing_proportions,  # Blood proportions
        [
            1 - mix_prop for mix_prop in pseudo_samples_mixing_proportions
        ],  # Neuron proportions
    ]
).T  # shape: (number of pseudo samples, number of cell types)

objective = cp.Minimize(cp.sum_squares(P @ F - M))
constraints = [cp.sum(F, axis=0) == 1]  # each row sums to 1 (constraint of P(s | c))
problem = cp.Problem(objective, constraints)
problem.solve()
F_solution = F.value
F_solution.round(2)

ValueError: shape mismatch: objects cannot be broadcast to a single shape.  Mismatch is between arg 0 with shape (4, 8) and arg 1 with shape (4, 7).

In [20]:
P.shape, M.shape

((4, 2), (4, 7))

In [21]:
len(all_signatures)

8

In [ ]:
comparison_results = pd.DataFrame(
    {
        ("Ground truth", "Blood-Mono+Macro"): sig_proportions_df["Blood-Mono+Macro"],
        ("Ground truth", "Neuron"): sig_proportions_df["Neuron"],
        ("Estimated Proportions", "Blood-Mono+Macro"): F_solution[0],
        ("Estimated Proportions", "Neuron"): F_solution[1],
    },
    index=all_signatures,
)
comparison_results.round(2)